# Lab 11 Tasks and Solutions: Linear Regression to Spatial Regression

**Total: 100 points.** Each task has a point value and an empty starter cell. **Remember to submit the HTML and PDF exports.**

Data: San Diego Airbnb listings (Inside Airbnb extract of 7 July 2016, from Rey, Arribas-Bel and Wolf, 2023). One row is one listing. The outcome is `log_price`, the natural log of the nightly price in US dollars. The ten predictors are in the list `X`. Shuffles use fixed seeds. Run the walkthrough first if you need the background.

| Task | Points | Topic |
|---|---|---|
| 1 | 10 | Simple regression |
| 2 | 10 | Test of the slope |
| 3 | 12 | Multiple regression |
| 4 | 8 | Dummy variables |
| 5 | 12 | Weights and Moran's I |
| 6 | 10 | Local Moran clusters |
| 7 | 12 | Neighborhood fixed effects |
| 8 | 14 | Spatial lag model |
| 9 | 12 | Choosing a model (open-ended) |

## Setup

The first cell prints the package versions. `mgwr` is installed if it is missing (it is only needed in the walkthrough).

In [1]:
import sys, subprocess, warnings
from importlib.metadata import version
warnings.filterwarnings("ignore")
try:
    import mgwr                           # geographically weighted regression, used in Part 8
except ImportError:                       # not installed in this kernel yet
    subprocess.check_call([sys.executable, "-m", "pip", "install", "mgwr"])
    import mgwr
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib
import matplotlib.pyplot as plt
import statsmodels.api as sm
import spreg
from scipy import stats
from libpysal import weights
from libpysal.weights import lag_spatial
from esda import Moran, Moran_Local

print("Python", sys.version.split()[0], "| geopandas", gpd.__version__, "| numpy", np.__version__,
      "| matplotlib", matplotlib.__version__)
print("spreg", version("spreg"), "| esda", version("esda"), "| libpysal", version("libpysal"),
      "| mgwr", version("mgwr"), "| statsmodels", version("statsmodels"))

Python 3.8.16 | geopandas 0.13.2 | numpy 1.24.3 | matplotlib 3.7.1
spreg 1.5.0 | esda 2.5.1 | libpysal 4.8.1 | mgwr 2.2.1 | statsmodels 0.14.0


In [2]:
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.lines import Line2D

PURPLE, MAGENTA, BLUE, GRAY = "#7C3AED", "#DB2777", "#2563EB", "#6B6478"
SEQ = LinearSegmentedColormap.from_list("seq", ["#F5F3FB", "#C4B5FD", PURPLE, MAGENTA])
DIV = LinearSegmentedColormap.from_list("div", [BLUE, "#F5F3FB", MAGENTA])
plt.rcParams.update({"font.size": 12, "axes.spines.top": False, "axes.spines.right": False})

# The ten predictors used in the lecture: guests, bathrooms, bedrooms, beds, room type, property type
X = ["accommodates", "bathrooms", "bedrooms", "beds", "rt_Private_room", "rt_Shared_room",
     "pg_Condominium", "pg_House", "pg_Other", "pg_Townhouse"]


def blank(ax):
    """Map style: no ticks, no frame, equal aspect."""
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect("equal")
    for s in ax.spines.values():
        s.set_visible(False)


def point_map(ax, gdf, values, cmap, norm, title, s=7):
    """Listings as colored dots. gdf needs columns px and py (meters)."""
    ax.scatter(gdf["px"], gdf["py"], c=values, cmap=cmap, norm=norm, s=s, alpha=0.85, linewidths=0)
    ax.set_title(title)
    blank(ax)


def coef_table(m, names):
    """Coefficients, standard errors and p-values of a spreg model, as a table (constant first)."""
    k = len(names)
    stat = m.t_stat if hasattr(m, "t_stat") else m.z_stat      # OLS reports t, the GMM models report z
    return pd.DataFrame({"coef": np.ravel(m.betas)[:k], "se": np.ravel(m.std_err)[:k],
                         "p": [t[1] for t in stat][:k]}, index=names)


def pfmt(p):
    """p-value as text: tiny values are shown as p < 0.001."""
    return "p < 0.001" if p < 0.001 else f"p = {p:.3f}"


def moran_of(u, w, perms=999, seed=0):
    """Moran's I of a vector u under weights w, with a fixed seed so the p-value repeats."""
    np.random.seed(seed)
    return Moran(np.ravel(u), w, permutations=perms)

In [ ]:
db = gpd.read_file("data/regression_db.geojson")
print(db.crs, db.shape)
db = db.to_crs(32611)                         # UTM zone 11N, meters: needed for distances and neighbors
db["px"], db["py"] = db.geometry.x, db.geometry.y
y = db[["log_price"]].values                  # the outcome as a column

ANS = {}                                      # answers are stored here by the solution cells

In [ ]:
w20 = weights.KNN.from_dataframe(db, k=20)    # each listing's 20 nearest listings
w20.transform = "R"                           # row-standardize: every row sums to 1
print(w20.n, "listings,", w20.mean_neighbors, "neighbors each, weight", w20.weights[0][0])

## Task 1 (10 points): Simple regression

Regress `log_price` on `accommodates` (the number of guests) with the least-squares formulas from Part 2 (or `np.polyfit`). Report the intercept, the slope and $R^2$. Say in one sentence what the slope means as a percent change in price.

In [ ]:
# your code here

## Task 2 (10 points): Test of the slope

For the regression in Task 1, compute by hand the standard error of the slope, the $t$ statistic and the two-sided p-value (use `scipy.stats.t.sf` with $n-2$ degrees of freedom). Check $t$ against `statsmodels`. State $H_0$ and your conclusion.

In [ ]:
# your code here

## Task 3 (12 points): Multiple regression

Fit log price on all ten predictors in `X` with `spreg.OLS` (as in Part 4). Report $R^2$, the coefficient on `bathrooms` and what it means as a percent change, and list the predictors whose p-value is above 0.05.

In [ ]:
# your code here

## Task 4 (8 points): Dummy variables

Using the model from Task 3: (a) name the baseline room type; (b) compute how many times as expensive a private room and a shared room are, compared with an entire home, holding everything else fixed; (c) in one sentence, say why the model does not include a dummy for the entire home.

In [ ]:
# your code here

## Task 5 (12 points): Weights and Moran's I

Using the residuals of the Task 3 model, compute Moran's I (999 shuffles, seed 0, with `moran_of`) with 20 nearest neighbors (`w20`) and with 10 nearest neighbors (`KNN` with `k=10`, row-standardized). Report both values and p-values, say which is larger and offer one reason, and say what the p-value tells you about the independence assumption.

In [ ]:
# your code here

## Task 6 (10 points): Local Moran clusters

Compute local Moran's I of the Task 3 residuals with `w20` (`np.random.seed(0)` before `Moran_Local(u1, w20, permutations=999)`). Count the listings that are high-high and the listings that are low-low at $p<0.01$, and report the share of each group that is coastal. Make a map with a legend and say what it tells you about the model.

In [ ]:
# your code here

## Task 7 (12 points): Neighborhood fixed effects

Fit the ten predictors plus one dummy per neighborhood and no separate intercept (`statsmodels`, as in Part 6). Report $R^2$, Moran's I of the residuals (`w20`, seed 0), the two neighborhoods with the highest intercepts and the two with the lowest, and how many times as expensive the same listing is predicted to be in the highest as in the lowest.

In [ ]:
# your code here

## Task 8 (14 points): Spatial lag model

Build row-standardized weights with 10 nearest neighbors (`w10` from Task 5) and fit the spatial lag model for the ten predictors with `spreg.GM_Lag`. Report $\rho$ and its standard error, Moran's I of the model residuals (`w10`, seed 0), the coefficient on `bedrooms`, and its total effect $\beta/(1-\rho)$. Say in a sentence why the total effect is larger than the coefficient.

In [ ]:
# your code here

## Task 9 (12 points): Choosing a model (open-ended)

A city wants to know whether a new condominium building raises the nightly prices of nearby short-term rentals, and wants honest standard errors for the other coefficients. Using the Lagrange multiplier tests, the SLX results for `pg_Condominium` (`w20`), the lag parameter and the error parameter, argue which model you would report and why. Write 4 to 6 sentences. There is no single correct model; you are graded on whether your argument uses the evidence and is consistent with how each model works.

In [ ]:
# your code here

## Optional (not graded)

Fit the geographically weighted regression from the walkthrough with `pool` added to the predictors. Does the model still fit? What does the failure or success tell you about GWR and dummy variables?